## Evaluation pipeline for the microlane experiment

In [1]:
# First consider all the variables
# The input image gets resized to a particular level
# Then create a pipeline to feed data into the model
# AFter this process is completed, then process the data
# Then after the processing is done find a way to take output from the model
# Then, convert the output to relevant format, and store it for future use
# Apply relevant computations

In [2]:
# Imports of the Core Packages
import yaml, random
from tqdm.notebook import tqdm

In [3]:
# Import custom libraries located at different folder location + configs
from microlane.utils.create_settings import create_settings
from microlane.utils.experiment import ExperimentEvaluate
from microlane.utils.loaders import load_dataset, load_model
from microlane.utils.load_config import load_config

In [4]:
config = load_config()

### Experiment Settings

In [ ]:
INFERENCE_VIS_NUMBER = 5
SAMPLE_NUMBER = 75
MODEL = "lanenet"
DATASET = "tusimple"
AUGMENTATION_TYPE = "normal"
EXPERIMENT_NAME = f"{MODEL}_{DATASET}_{AUGMENTATION_TYPE}"
AUGMENTATION_SETTINGS = {}
OUTPUT_LOCATION = "/home/suyog/desktop/projects/microlane/results/EXPERIMENT/raw/lanenet/motion-blur"


### Pre Processing Part

In [10]:
## Load the Corresponding Model and the dataset based on given settings

data  = load_dataset(DATASET, SAMPLE_NUMBER)

model = load_model(MODEL)


Config(pipeline=PipelineConfig(name='Microlane', version='1.0.0', description='Evaluating how well modern lane detection models hold up when deployed on small-scale 1/10 RC cars', default_port=8000), data=DataConfig(datasets=DatasetsConfig(tusimple=DatasetConfig(name='tusimple', path=PosixPath('/home/suyog/assets/datasets/TuSimple/TUSimple'), annotation_file=PosixPath('/home/suyog/assets/datasets/TuSimple/test_label_path_normalised.json'), enabled=True), custom_dataset=DatasetConfig(name='custom_dataset', path=PosixPath('/home/suyog/desktop/projects/microlane/results/custom_dataset'), annotation_file=PosixPath('/home/suyog/desktop/projects/microlane/results/custom_dataset/annotations.xml'), enabled=True)), augmentation=AugmentationConfig(ranges=AugmentationRangesConfig(motion_blur_range=(0.0, 0.1), lighting_range=(-1.0, 1.0), blur_range=(0.0, 1.0), zoom_range=(1.0, 3), rotation_range=(0.0, 360.0)), presets={'normal': {'blur': 0.0, 'rotation': 0.0, 'zoom': 1.0, 'lighting': 0.0, 'motion_

In [ ]:
experiment = ExperimentEvaluate(
    
    experiment_name=EXPERIMENT_NAME,
    
    output_dir=OUTPUT_LOCATION
    
)

### Models and Datasets Loaded, Now Processing Part

In [12]:
# Print some basic information of our data

print(f"Total items: {len(data)}\n")

item = data[0]
print(f"Image Path   : {item.image_path}")
print(f"h_samples    : {item.h_samples}")
print(f"lanes        : {item.lanes}")

Total items: 0



IndexError: list index out of range

In [ ]:
numbers = [random.randint(0, SAMPLE_NUMBER - 1) for _ in range(INFERENCE_VIS_NUMBER)]

In [ ]:
settings = create_settings(
    inference_vis_number=INFERENCE_VIS_NUMBER,
    sample_number=SAMPLE_NUMBER,
    model=MODEL,
    dataset=DATASET,
    augmentation_type=AUGMENTATION_TYPE,
    augmentation_settings=AUGMENTATION_SETTINGS,
    output_path=experiment.folder_dir,
    experiment_name=EXPERIMENT_NAME,
)

### Looping through all the testing examples

In [ ]:
for index, testing_example in enumerate(tqdm(data)):

    testing_example.blur        = AUGMENTATION_SETTINGS["blur"]
    testing_example.rotation    = AUGMENTATION_SETTINGS["rotation"]
    testing_example.zoom        = AUGMENTATION_SETTINGS["zoom"]
    testing_example.lighting    = AUGMENTATION_SETTINGS["lighting"]
    testing_example.motion_blur = AUGMENTATION_SETTINGS["motion_blur"]
    
    if AUGMENTATION_SETTINGS['shake']:
        
        testing_example.rotation = random.uniform(-10.0, 10.0)   # degrees
        
        testing_example.zoom     = random.uniform(1, 1.15)  # subtle scale jitter
        

    loaded_image = load_image_from_sample(testing_example)
    
    response = model.predict(loaded_image)
    
    modeloutput = parse_prediction(response)
    
    experiment.store_prediction(modeloutput)
        
    if index in numbers:
        
        experiment.visualize_prediction(modeloutput)
    
    if (index % 100 == 0) and (index != 0):
        
        print(f"Routine Container Restart for Addressing Memory Leak [{int(index/100)}]")
        
        model.container_manager.restart_container()